# STAC >> MinIO << Full

In [ ]:
import sys
sys.path.insert(1, '../utils/')

In [ ]:
# reload module before executing code
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Standard library imports
import os
import re
from copy import deepcopy
# from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# Third-party library imports
import requests
import pystac_client
import planetary_computer as pc
from IPython.display import HTML, JSON
from tqdm import tqdm

# Local module imports
from utils.nostra_add_product import (
    select_and_save_product,
    add_product_via_cli,
    parse_product_yaml
)
from utils.nostra_mapping import create_map, last_drawn_rectangle
from utils.nostra_minio import (
    connect_and_check_endpoint,
    create_bucket,
    set_anonymous_download_permissions,
    list_bucket_tree,
    empty_bucket,
    delete_bucket,
)
from utils.nostra_stac_to_minio import (
    stac_to_minio,
    list_last_level_folders,
    valid_folders,
    prepare_folders,
    extract_url,
    index_minio_yamls,
)

## 1. Configure and add a new product

In [ ]:
# Configuration

yamls_dict = {
        "local": "/conf",
        "dc_tools": "https://raw.githubusercontent.com/opendatacube/odc-tools/refs/heads/develop/apps/dc_tools/tests/data/example_product_list.csv"
    }

Before indexing, a product description in YAML format needs to be added to the datacube.

The following cell generates a YAML template based on a selection of existing products. You will need to adapt this template to the product you want to add before proceeding with the subsequent cells.

In [ ]:
select_and_save_product(yamls_dict, "example.yaml")

In [ ]:
# adapted yaml

yaml_file = 'ls89_c2l2_sp_local-product.yaml'
parse_product_yaml(yaml_file)

In [ ]:
# TbD: Print variable and values to be found on yaml

r = add_product_via_cli(yaml_file, update=True)

## 2. Identify, download and transfer scenes files on MinIO

In [ ]:
m = create_map((-180, -90, 180, 90))
display(m)

In [ ]:
aoi_poly = last_drawn_rectangle
if len (aoi_poly) == 0:
    # Change the background color of the output area to light red
    display(HTML('''
        <div style="background-color: red; padding: 10px; border-radius: 5px;">
            The area of interest polygon (aoi_poly) has not been created. Please draw it in the previous cell.
        </div>
    '''))
else:

    aoi_poly = aoi_poly.get('bounds')

In [ ]:
# DEV: by pass manual AoI drawing

aoi_poly = (7.001038, 46.871708, 7.176819, 46.98425)

In [ ]:
# Configuration

stac_endpoint = "https://planetarycomputer.microsoft.com/api/stac/v1"
stac_collection = "landsat-c2-l2"
stac_platforms=["landsat-8", "landsat-9"]
stac_time = ('2024-02-15', '2024-04-01')  # end date is not inclusive !
stac_layers = ['red', 'green', 'blue'] # all if empty ['atran', 'blue', 'cdist', 'coastal', 'drad', 'emis', 'emsd', 'green', 'lwir11', 'nir08', 'qa', 'qa_aerosol', 'qa_pixel', 'qa_radsat', 'red', 'swir16', 'swir22', 'trad', 'urad']

# product_type = 'Landsat'
product_name = 'ls89_c2l2_sp_local'  # 'landsat_ot_c2_l2'
yaml_file = 'landsat89_mtl_config.yaml'

minio_endpoint = "minio:9000"
minio_https = False
minio_access_key = "admin"
minio_secret_key = "admin123"
minio_bucket = 'test'

minio_url = "http://minio:9000"  # "http://localhost:9000"

In [ ]:
# connect to MiniIO

minio_client = connect_and_check_endpoint(minio_endpoint, minio_access_key, minio_secret_key, minio_https)
create_bucket(minio_client, minio_bucket)
r = set_anonymous_download_permissions(minio_client, minio_bucket)

In [ ]:
%%time

# Download from STAC to MinIO

assets_dict, download_results = stac_to_minio(
    minio_client = minio_client, minio_bucket = minio_bucket,
    stac_endpoint = stac_endpoint, stac_collection = stac_collection,
    platforms=stac_platforms,
    stac_time = stac_time, stac_layers = stac_layers, t1_only = True,
    aoi_poly = aoi_poly,
    overwrite=False, complete=True, dry_run=False, verbose=True
)
JSON(download_results) # display summary

In [ ]:
# Optional

list_bucket_tree(minio_client, minio_bucket, 'landsat-c2/level-2/standard/oli-tirs',
                 max_recursion_level=4, show_sizes=True)

## Prepare yamls now !

In [ ]:
# optionally remove existing yamls

r = empty_bucket(minio_client, minio_bucket, pattern="*.yaml")

In [ ]:
# List scenes to process

# from stac_to_minio funstion
folders = list(assets_dict.keys())

# # all folders on MinIO
# folders = list_last_level_folders(minio_client, minio_bucket)  # filter_pattern="oli-tirs/**/*202406*202406"

In [ ]:
# Check if yaml already exist and keep if overwrite=True

folders = valid_folders(minio_client, minio_bucket, deepcopy(folders), overwrite=True)

In [ ]:
results = prepare_folders(folders, minio_client, minio_bucket, minio_url, product_name, yaml_file)

In [ ]:
results

## Index scenes

In [ ]:
%%time

yamls =[extract_url(r['result']) for r in results]
execution_results, errors = index_minio_yamls(yamls, product_name, max_workers=4)

# Summary
successful = len([r for r in execution_results if r["success"]])
failed = len(errors)

print(f"\n--- Execution Summary ---")
print(f"Total processed: {len(execution_results)}")
print(f"Successful: {successful}")
print(f"Failed: {failed}")

# Display errors if any
if errors:
    print(f"\n--- Errors ({len(errors)}) ---")
    for error in errors:
        print(f"URL: {error['url']}")
        print(f"Error: {error['error']}")
        if 'returncode' in error:
            print(f"Return code: {error['returncode']}")
        print("-" * 50)

In [ ]:
%%time

# Optional AND DANGEROUS ⚠️

r = empty_bucket(minio_client, minio_bucket)
r = delete_bucket(minio_client, minio_bucket)